# Model Training

This notebook trains and evaluates machine learning models for predictive maintenance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.append('../src')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully')

## 1. Load Engineered Data

In [ ]:
# Load engineered dataset
df = pd.read_csv('../data/processed/ai4i2020_engineered.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Select Features

In [ ]:
# Select feature columns (exclude non-predictive columns)
exclude_cols = ['udi', 'product_id', 'machine_failure', 'type', 
               'twf', 'hdf', 'pwf', 'osf', 'rnf', 'health_grade',
               'noise_flags']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Total features selected: {len(feature_cols)}")
print(f"\nFeature columns:\n{feature_cols}")

## 3. Initialize Model Trainer

In [ ]:
from train_model import ModelTrainer

# Initialize trainer
trainer = ModelTrainer(random_state=42)
print('Model trainer initialized')

## 4. Split Data

In [ ]:
# Split data into train, validation, and test sets
X_train, X_val, X_test, y_train, y_val, y_test = trainer.split_data(
    df=df,
    feature_cols=feature_cols,
    target_col='machine_failure',
    test_size=0.2,
    validation_size=0.1
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

## 5. Train Random Forest

In [ ]:
# Train Random Forest classifier
rf_model = trainer.train_random_forest(X_train, y_train, hyperparameter_tuning=False)
print("Random Forest trained successfully")

## 6. Train Gradient Boosting

In [ ]:
# Train Gradient Boosting classifier
gb_model = trainer.train_gradient_boosting(X_train, y_train, hyperparameter_tuning=False)
print("Gradient Boosting trained successfully")

## 7. Train XGBoost (Optional)

In [ ]:
# Try to train XGBoost if available
try:
    xgb_model = trainer.train_xgboost(X_train, y_train, hyperparameter_tuning=False)
    if xgb_model:
        print("XGBoost trained successfully")
except:
    print("XGBoost not available or failed to train")

## 8. Evaluate Models

In [ ]:
# Collect trained models
trained_models = {
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model
}

# Add XGBoost if trained
try:
    if xgb_model:
        trained_models['XGBoost'] = xgb_model
except:
    pass

# Compare models
comparison_df = trainer.compare_models(trained_models, X_test, y_test)
print("\nModel Comparison:")
print(comparison_df)

## 9. Feature Importance

In [ ]:
# Get feature importance for best model
best_model_name = comparison_df['roc_auc'].idxmax()
best_model = trained_models[best_model_name]

importance_df = trainer.get_feature_importance(best_model, feature_cols, top_n=15)

# Plot feature importance
if importance_df is not None:
    plt.figure(figsize=(12, 8))
    sns.barplot(data=importance_df.head(15), x='importance', y='feature')
    plt.title(f'Top 15 Feature Importances - {best_model_name}')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

## 10. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 8))

for model_name, model in trained_models.items():
    if model is None:
        continue
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.show()

## 11. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# Plot confusion matrix for best model
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 12. Run Full Training Pipeline

In [ ]:
# Run complete training pipeline
results = trainer.full_training_pipeline(
    df=df,
    feature_cols=feature_cols,
    target_col='machine_failure',
    models_to_train=['rf', 'gb'],
    save_path='../models'
)

print(f"\nBest model: {results['best_model_name']}")
print(f"ROC-AUC: {results['comparison'].loc[results['best_model_name'], 'roc_auc']:.4f}")

## 13. Save Feature Columns

In [ ]:
import json

# Save feature columns for prediction
with open('../models/feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)

print(f"Feature columns saved to ../models/feature_columns.json")

## Summary

This notebook performed:
- Data loading and feature selection
- Train/validation/test split
- Random Forest model training
- Gradient Boosting model training
- XGBoost model training (if available)
- Model comparison and evaluation
- Feature importance analysis
- ROC curve visualization
- Confusion matrix analysis
- Full training pipeline execution
- Model and feature column export